# 02 - EDA dirigido a la problemática (es_incapacitante)

A diferencia de `01_eda_accidentes.ipynb` (descriptivo, sobre 2023-2024),
este notebook trabaja **solo sobre `accidentes_historico_2012_2022.csv`**
(única fuente con `gravedad`) y evalúa las X candidatas contra la variable
objetivo `es_incapacitante` (ver README, secciones 4-5, y
`reports/diccionario_datos.md`).

La lógica vive en `src/clean.py` (`derive_es_incapacitante`,
`normalize_turno`) y `src/eda.py` (`cramers_v`,
`tasa_incapacitante_por_categoria`) — este notebook solo importa y muestra.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.clean import derive_es_incapacitante
from src.ingest import load_historico_accidentes

df = load_historico_accidentes()
y = derive_es_incapacitante(df["gravedad"])

print("shape df:", df.shape)
print()
print("y (es_incapacitante) value_counts, incluye NA:")
print(y.value_counts(dropna=False))
print()
n_definido = y.notna().sum()
print(f"Filas con etiqueta definida: {n_definido} de {len(df)}")
print("Distribucion sobre las filas con etiqueta definida:")
print((y.dropna().value_counts(normalize=True) * 100).round(2))

shape df: (1028, 27)

y (es_incapacitante) value_counts, incluye NA:
gravedad
True     861
False    155
<NA>      12
Name: count, dtype: Int64

Filas con etiqueta definida: 1016 de 1028
Distribucion sobre las filas con etiqueta definida:
gravedad
True     84.74
False    15.26
Name: proportion, dtype: Float64


## Correlación numérica (Pearson)

Solo 2 columnas de X son numéricas: `edad_anios`, `anio`. `mes` es
categórica en esta fuente (nombre del mes en texto, ej. "ABRIL"), no
numérica — se evalúa más abajo junto con las demás categóricas.
`experiencia_puesto` quedó **fuera de X** en esta fase (texto libre con
unidades mezcladas — ver README sección 5); no se incluye aquí.

In [2]:
cols_numericas = ["edad_anios", "anio"]
df[cols_numericas].corr()

,edad_anios,anio
edad_anios,1.000000,0.002074
anio,0.002074,1.000000


## Asociación categórica (Cramér's V) contra `es_incapacitante`

Cramér's V va de 0 (sin asociación) a 1 (asociación perfecta) — el
equivalente a una correlación para variables categóricas, donde Pearson no
aplica. Se separan las X de cardinalidad manejable de las de cardinalidad
muy alta (ver nota abajo).

In [3]:
from src.eda import cramers_v

cols_categoricas_ok = [
    "sexo", "turno", "mes", "programa", "area_responsabilidad", "actividad_realizada"
]

resultado = {col: round(cramers_v(df[col], y), 4) for col in cols_categoricas_ok}
pd.Series(resultado, name="cramers_v").sort_values(ascending=False)

actividad_realizada     0.6401
programa                0.5705
area_responsabilidad    0.3440
turno                   0.2709
mes                     0.2127
sexo                    0.1589
Name: cramers_v, dtype: float64

### X de cardinalidad muy alta: `lugar`, `fuente_peligro`, `puesto_trabajo`

**Cramér's V poco confiable por celdas con n<5; interpretar solo el orden
de magnitud.** Son prácticamente texto libre (344-371 categorías sobre 1028
filas) más que categóricas — agruparlas requiere criterio de dominio que no
tenemos hoy (ver README, sección 9, pendiente).

In [4]:
cols_categoricas_alta_cardinalidad = ["lugar", "fuente_peligro", "puesto_trabajo"]

for col in cols_categoricas_alta_cardinalidad:
    v = round(cramers_v(df[col], y), 4)
    print(f"{col}: {df[col].nunique()} categorias -> Cramer's V = {v}")
    print("  (poco confiable por celdas con n<5; interpretar solo el orden de magnitud)")

lugar: 371 categorias -> Cramer's V = 0.8197
  (poco confiable por celdas con n<5; interpretar solo el orden de magnitud)
fuente_peligro: 359 categorias -> Cramer's V = 0.8419
  (poco confiable por celdas con n<5; interpretar solo el orden de magnitud)
puesto_trabajo: 344 categorias -> Cramer's V = 0.8077
  (poco confiable por celdas con n<5; interpretar solo el orden de magnitud)


## Tasa de incapacitantes por categoría (tablas de contingencia)

Para las X de cardinalidad manejable: cuántos casos y qué % es
incapacitante, por categoría.

In [5]:
from src.eda import tasa_incapacitante_por_categoria

for col in cols_categoricas_ok:
    print(f"--- {col} ---")
    display(tasa_incapacitante_por_categoria(df[col], y))
    print()

--- sexo ---


,n,pct_incapacitante
categoria,,
M,572,81.47
F,98,65.31
-,1,0.0



--- turno ---


,n,pct_incapacitante
categoria,,
DIA,323,81.11
TARDE,220,73.64
NOCHE,136,79.41
TURNO_1,119,100.0
TURNO_2,76,100.0
TURNO_3,45,100.0



--- mes ---


...

,n,pct_incapacitante
categoria,,
GALLETERA LIMA,115,56.52
COPSA,70,68.57
TEAL,63,98.41
SI,61,100.0
DISTRIBUCION LIMA,36,69.44
...,...,...
ITDC-LUIRN 4,1,100.0
ITDC-DETERGENTES,1,100.0
ITDC-CHORRILLOS,1,0.0



--- area_responsabilidad ---


,n,pct_incapacitante
categoria,,
PRODUCCION,252,93.25
DISTRIBUCION,64,100.0
ADMINISTRACION,36,80.56
GLOBAL,34,100.0
PROYECTOS,28,100.0
MANTENIMIENTO,27,96.3
VITAPRO,13,100.0
ALMACEN DE INSUMOS,13,84.62
RELACIONES LABORALES,13,100.0



--- actividad_realizada ---


,n,pct_incapacitante
categoria,,
DURANTE OPERACION PRODUCTIVA,282,87.23
DURANTE SU RUTINA DE TRABAJO,212,99.06
OTROS,75,45.33
TRASLADANDOSE A PIE,68,75.0
MANTENIMIENTO,60,98.33
...,...,...
CONDUCCION DE MONTACARGA,1,0.0
CARGA MATERIALES,1,100.0
CARGA ADITIVOS,1,100.0


### `lugar`, `fuente_peligro`, `puesto_trabajo`: mismo aviso de arriba

**Cramér's V poco confiable por celdas con n<5; interpretar solo el orden
de magnitud.** Se muestra solo el top 10 por cantidad de casos, para no
listar cientos de categorías con 1-2 filas cada una.

In [6]:
for col in cols_categoricas_alta_cardinalidad:
    print(f"--- {col} (top 10 por n) ---")
    display(tasa_incapacitante_por_categoria(df[col], y).head(10))
    print()

--- lugar (top 10 por n) ---


,n,pct_incapacitante
categoria,,
COPSA,61,85.25
GALLETERA LIMA,45,95.56
DETERGENTES,42,88.1
GLOBAL,36,100.0
GALLETERA,28,28.57
NUTRICION ANIMAL TRUJILLO,27,100.0
SIDSUR,26,100.0
FIDEERIA LIMA,25,92.0
CDC,22,90.91



--- fuente_peligro (top 10 por n) ---


,n,pct_incapacitante
categoria,,
EQUIPO,122,100.0
PISO,55,87.27
OTROS,53,11.32
MAQUINA,38,44.74
CAMION,29,96.55
FAJA EN MOVIMIENTO,23,100.0
ESCALERA,23,91.3
PALETA,22,90.91
PUERTA,17,94.12



--- puesto_trabajo (top 10 por n) ---


,n,pct_incapacitante
categoria,,
PERSONAL ALICORP,100,100.0
TERCERO,88,100.0
OPERARIO DE PRODUCCION,63,90.48
AYUDANTE DE PLANTA,56,85.71
PERSONAL NO ALICORP,44,100.0
OPERARIO,31,51.61
OBRERO,30,16.67
MECANICO,18,100.0
PERSONAL TEAL,14,100.0


## Pendiente

- [ ] Este notebook es descriptivo/exploratorio: los resultados de arriba
      sirven para descartar X redundantes, no miden importancia — eso lo
      dará el modelo en fase 2 (ver README sección 7).
- [ ] `lugar`, `fuente_peligro`, `puesto_trabajo` necesitan una regla de
      agrupación por criterio de dominio antes de usarse con confianza en un
      modelo — no se decidió en esta pasada.
- [ ] `experiencia_puesto` sigue fuera de X (texto libre, unidades
      mezcladas) — normalizarla es una fase futura.
- [ ] Decidir el caso a caso de las 2 filas de `gravedad` pendientes
      ("ACCIDENTE FUERA DEL TRABAJO", "DAÑO A LA SALUD") — hoy quedan como
      `es_incapacitante = NA`, excluidas de este análisis.